In [ ]:
import pandas as pd
from rapidfuzz import fuzz, process

In [ ]:
# load full metadata
df = pd.read_excel('merged_journals_metadata.xlsx')


In [ ]:
df.info()

In [ ]:
keywords = (df['subject']
                .dropna()
                .str.replace('; ', ' | ', regex=False)
                .str.replace('；', ' | ', regex=False)
                .str.replace(', ', ' | ', regex=False)
                .str.replace('，', ' | ', regex=False)
                .str.split(' | ', regex=False, expand=False) 
                .explode()
                .dropna()
                .str.strip('| .,;')
                .str.lower())

keywords_stats_multilang = keywords.value_counts()

In [ ]:
with open('translated_keywords.txt', 'r', encoding='utf-8') as txt:
    translated_keywords = [e.strip() for e in txt.readlines()]
translated_keywords = [e.split(' [--translated-->] ')[1] for e in translated_keywords]

with open('unique_keywords.txt', 'r', encoding='utf-8') as txt:
    unique_keywords = [e.strip() for e in txt.readlines()]
    
translation_map = dict(zip(unique_keywords, translated_keywords))

keywords_translated = keywords.map(translation_map)

keywords_stats_en = keywords_translated.value_counts()

keywords_df = pd.DataFrame({
    "original": keywords,
    "translated": keywords_translated,
})

keywords_df['translated'] = keywords_df['translated'].str.lower()
keywords_df = (
    keywords_df.dropna(subset=['translated'])
      .reset_index()
      .groupby(['index', 'translated'])
      .agg(originals=('original', lambda x: set(x)))
      .reset_index()
)

keywords_grouped = (
    keywords_df
    .groupby('translated')
    .agg(
        total_count=('translated', 'count'),
        originals_global=('originals', lambda x: list({o for s in x for o in s}))  
    )
    .sort_values(by='total_count', ascending=False)
    .reset_index()
)
keywords_grouped.head(10)

In [ ]:
# authors stats
authors_stats = (df['creator']
                .dropna()
                .str.split(' | ', regex=False, expand=False) 
                .explode()
                .dropna()
                .str.strip())
authors_stats = pd.DataFrame(authors_stats)
affiliations_to_replace = ['(Chulalongkorn University, THAILAND), and ',
'(University of Manchester, UNITED KINGDOM), ',
'(Yarmouk University), ',
'(Jerash University), ',
'(Yarmouk University, JORDAN), ',
'(Beihang University, CHINA), ',
'(University of Extremadura, SPAIN), ',
'(University of Granada, SPAIN), ',
'(Universitat de València, SPAIN), ',
'(Leipzig University, GERMANY), ',
'(Guizhou Normal University, CHINA), ',
'(University Putra Malaysia, MALAYSIA; University of Newcastle, AUSTRALIA), ',
'(University Putra Malaysia, MALAYSIA), and ',
'(The John Paul II Catholic University of Lublin, POLAND), ',
'(Minia University, EGYPT), ',
'(University College London, UNITED KINGDOM), ',
'(Kent State University, UNITED STATES OF AMERICA), ',
'(The University of Western Australia, AUSTRALIA), ',
'(Queen’s University Belfast, UNITED KINGDOM), ',
'(National Police Academy, Ankara, TÜRKIYE), ',
'(Moscow City Teacher Training University, RUSSIA), ',
'(Universidad de Granada, SPAIN), ',
'(Hamad Bin Khalifa University, QATAR), ',
'(Leipzig University, GERMANY), and ',
'(University of Zadar/University of Applied Sciences Velika Gorica, CROATIA), ',
'(Kent State University, USA), ',
'(Hong Kong Baptist University, HONG KONG SAR), ',
'(Northwest Normal University/City University of Hong Kong), ',
'(City University of Hong Kong, HONG KONG SAR), ',
'(Politeknik El Bajo Commodus, INDONESIA), ',
'(Independent researcher, HONG KONG SAR), ',
', Universidad de Córdoba',]

for frag in affiliations_to_replace:
    authors_stats['creator'] = authors_stats['creator'].str.replace(frag, "", regex=False)

authors_field_to_drop = ['Tradução, Cadernos de',
'de Tradução, Cadernos',
'New Voices Editorial Team',
'Clina, Secretaría de Redacción',
'de redacción, Consello',
'Tradução, Cadernos',
'Notícias, Transfer',
'Autores, Varios',
'Transfer',
'UB, Transfer',
'Sendebar, Revista',
'Reviews, Book',
'Varios, Varios',
'Editorial, Equipo',
'de Traducción, Estudios',
'anonimo, anonimo',
'Equipo Editorial',
'Clina, Secretaría de redacción',
'Equipo editorial de Estudios de Traducción',
'TTS, LANS',
'Reviews, All',
'Sedebar, Revista',
'CLINA, Secretaría de redacción',
'de Filoloxía e Tradución, Facultade',
'editorial, Equipo ',
'Equipo Editorial Onmazein',
'Estudios de Traducción, Revista',
'de Traduccion, Estudios',
'Consejo de Redacción',
'CONSEJO EDITORIAL',
'Editorial Team, New Voices',
'New Voices Editorial Team and Guest Editors',
'Editorial Team, New Voices ',
'Issue, Complete',
'Special Issue: Intersemiotic Translation and Multimodality, Translation Matters',
'editorial, Equip',
'«Ticontre», Readazione',
'News, Transfer',
'Cret, Transfer',
'Universidad De Salamanca, Ediciones',
'Secretaría de redacción, CLINA',
'CLINA Secretaría de redacción',
'editorial, Equipo',
'Revista, Sendebar',
'Cadernos de Tradução, Cadernos',
'Varios, Autores',
'Varios, varios',
'CT, Os Editores',
'Cadernos, Editores',]

authors_stats = authors_stats[~authors_stats['creator'].isin(authors_field_to_drop)]

authors_stats = (authors_stats['creator'].value_counts())

print(authors_stats[:10])

In [ ]:
# authors grouping based on similarity
unique_names = authors_stats.index.tolist()

groups = []
used = set()

for name in unique_names:
    if name in used:
        continue
    
    group = [name]
    used.add(name)
    
    matches = process.extract(
        name,
        unique_names,
        scorer=fuzz.token_sort_ratio,
        score_cutoff=90
    )
    
    for match, score, _ in matches:
        if match not in used:
            group.append(match)
            used.add(match)
    
    groups.append(group)

authors_canonical_rows = []
for group in groups:
    canonical = max(group, key=lambda x: authors_stats[x])
    
    total_count = sum(authors_stats[n] for n in group)
    
    authors_canonical_rows.append({
        "canonical_name": canonical,
        "variants": group,
        "count": total_count
    })



In [ ]:
authors_similarity_df = pd.DataFrame(authors_canonical_rows)
authors_similarity_df.head()

In [ ]:
# multiple authors stats
multiple_authors_stats = df['creator'].dropna()

multiple_authors_stats = pd.DataFrame(multiple_authors_stats)
affiliations_to_replace = ['(Chulalongkorn University, THAILAND), and ',
'(University of Manchester, UNITED KINGDOM), ',
'(Yarmouk University), ',
'(Jerash University), ',
'(Yarmouk University, JORDAN), ',
'(Beihang University, CHINA), ',
'(University of Extremadura, SPAIN), ',
'(University of Granada, SPAIN), ',
'(Universitat de València, SPAIN), ',
'(Leipzig University, GERMANY), ',
'(Guizhou Normal University, CHINA), ',
'(University Putra Malaysia, MALAYSIA; University of Newcastle, AUSTRALIA), ',
'(University Putra Malaysia, MALAYSIA), and ',
'(The John Paul II Catholic University of Lublin, POLAND), ',
'(Minia University, EGYPT), ',
'(University College London, UNITED KINGDOM), ',
'(Kent State University, UNITED STATES OF AMERICA), ',
'(The University of Western Australia, AUSTRALIA), ',
'(Queen’s University Belfast, UNITED KINGDOM), ',
'(National Police Academy, Ankara, TÜRKIYE), ',
'(Moscow City Teacher Training University, RUSSIA), ',
'(Universidad de Granada, SPAIN), ',
'(Hamad Bin Khalifa University, QATAR), ',
'(Leipzig University, GERMANY), and ',
'(University of Zadar/University of Applied Sciences Velika Gorica, CROATIA), ',
'(Kent State University, USA), ',
'(Hong Kong Baptist University, HONG KONG SAR), ',
'(Northwest Normal University/City University of Hong Kong), ',
'(City University of Hong Kong, HONG KONG SAR), ',
'(Politeknik El Bajo Commodus, INDONESIA), ',
'(Independent researcher, HONG KONG SAR), ',
', Universidad de Córdoba',]

for frag in affiliations_to_replace:
    multiple_authors_stats['creator'] = multiple_authors_stats['creator'].str.replace(frag, "", regex=False)

authors_field_to_drop = ['Tradução, Cadernos de',
'de Tradução, Cadernos',
'New Voices Editorial Team',
'Clina, Secretaría de Redacción',
'de redacción, Consello',
'Tradução, Cadernos',
'Notícias, Transfer',
'Autores, Varios',
'Transfer',
'UB, Transfer',
'Sendebar, Revista',
'Reviews, Book',
'Varios, Varios',
'Editorial, Equipo',
'de Traducción, Estudios',
'anonimo, anonimo',
'Equipo Editorial',
'Clina, Secretaría de redacción',
'Equipo editorial de Estudios de Traducción',
'TTS, LANS',
'Reviews, All',
'Sedebar, Revista',
'CLINA, Secretaría de redacción',
'de Filoloxía e Tradución, Facultade',
'editorial, Equipo ',
'Equipo Editorial Onmazein',
'Estudios de Traducción, Revista',
'de Traduccion, Estudios',
'Consejo de Redacción',
'CONSEJO EDITORIAL',
'Editorial Team, New Voices',
'New Voices Editorial Team and Guest Editors',
'Editorial Team, New Voices ',
'Issue, Complete',
'Special Issue: Intersemiotic Translation and Multimodality, Translation Matters',
'editorial, Equip',
'«Ticontre», Readazione',
'News, Transfer',
'Cret, Transfer',
'Universidad De Salamanca, Ediciones',
'Secretaría de redacción, CLINA',
'CLINA Secretaría de redacción',
'editorial, Equipo',
'Revista, Sendebar',
'Cadernos de Tradução, Cadernos',
'Varios, Autores',
'Varios, varios',
'CT, Os Editores',
'Cadernos, Editores',]

multiple_authors_stats = multiple_authors_stats[~multiple_authors_stats['creator'].isin(authors_field_to_drop)]
multiple_authors_stats = multiple_authors_stats['creator'].dropna().str.count(r'\|').add(1).value_counts()

print(multiple_authors_stats[:10])

In [ ]:
# language stats
lang_stats = (df['language']
                .dropna()
                .str.split(' | ', regex=False, expand=False) 
                .explode()
                .str.replace(r'^en$', 'eng', regex=True)
                .str.replace(r'^es$', 'spa', regex=True)
                .dropna()
                .value_counts())

print(lang_stats[:10])

In [ ]:
# dates stats
dates_stats = (df['date']
        .dropna()
        .str.split(' | ', regex=False, expand=False) 
        .explode()
        .dropna()
        .str.replace(r'-.+$', '', regex=True)
        .value_counts())
print(dates_stats[:10])

earliest = df['date'].min()
latest = df['date'].max()
print(earliest, latest)

df_years_kwds = df.copy()
df_years_kwds["year"] = pd.to_datetime(df_years_kwds["date"]).dt.year
df_years_kwds["subject"] = (df_years_kwds["subject"]
                                .str.replace('; ', ' | ', regex=False)
                                .str.replace('；', ' | ', regex=False)
                                .str.replace(', ', ' | ', regex=False)
                                .str.replace('，', ' | ', regex=False)
                                .str.split(' | ', regex=False, expand=False))
df_years_kwds = df_years_kwds.explode("subject")
df_years_kwds["subject"] = (
    df_years_kwds["subject"]
    .str.strip('| .,;')
    .str.lower()
)

with open('translated_keywords.txt', 'r', encoding='utf-8') as txt:
    keywords_mapping = [e.strip() for e in txt.readlines()]
keywords_mapping = {e.split(' [--translated-->] ')[0]:e.split(' [--translated-->] ')[1] for e in keywords_mapping}
df_years_kwds["subject"] = df_years_kwds["subject"].replace(keywords_mapping).str.lower().str.strip()
df_years_kwds = df_years_kwds.drop_duplicates(subset=["oai_id", "subject"], keep="first")
df_years_kwds_grouped = df_years_kwds.dropna(subset=["subject"]).groupby("year")["subject"].value_counts()

print(df_years_kwds_grouped.head(10))

In [ ]:
with pd.ExcelWriter('metadata_stats.xlsx') as writer:
    keywords_stats_multilang.reset_index().rename(columns={'index':'keyword'}).to_excel(writer, sheet_name='keywords_stats_multilang', index=False)
    keywords_stats_en.reset_index().rename(columns={'index':'keyword'}).to_excel(writer, sheet_name='keywords_stats_en', index=False)
    keywords_grouped.to_excel(writer, sheet_name='keywords_grouped', index=False)
    authors_stats.reset_index().rename(columns={'index':'author'}).to_excel(writer, sheet_name='authors_stats', index=False)
    authors_similarity_df.to_excel(writer, sheet_name='authors_similarity', index=False)
    multiple_authors_stats.reset_index().rename(columns={'index':'authors_number'}).to_excel(writer, sheet_name='multiple_authors_stats', index=False)
    lang_stats.reset_index().rename(columns={'index':'language'}).to_excel(writer, sheet_name='lang_stats', index=False)
    df_years_kwds_grouped.reset_index().rename(columns={'index':'year'}).to_excel(writer, sheet_name='years_kwds_grouped', index=False)